# 结合 GCP Functions 与 ChatGPT 中的 GPT Actions 使用 GCP BigQuery

本 notebook 分步介绍如何将 Google Cloud BigQuery 用作具备向量搜索能力的数据库，并结合 OpenAI 嵌入使用，然后在其之上创建一个 Google Cloud Function，以接入 ChatGPT 中的自定义 GPT（Custom GPT）。

对于希望在 Google Cloud Platform（GCP）内部搭建 RAG 基础设施、并将其作为端点对外暴露以便与 ChatGPT 等其他平台集成的客户来说，这可以作为一种解决方案。

Google Cloud BigQuery 是一个全托管、无服务器的数据仓库，借助 Google 基础设施的处理能力实现超快速的 SQL 查询。它让开发者可以轻松存储和分析海量数据集。

Google Cloud Functions 是一种轻量级、基于事件的异步计算解决方案，让你无需管理服务器或运行时环境，就能创建响应云事件的小型单一用途函数。

## 前提条件：

要运行本 cookbook，你需要具备：
- 一个你有权访问的 GCP 项目
- 具有创建 BigQuery 数据集和 Google Cloud Function 权限的 GCP 用户
- 已安装并连接的 [GCP CLI](https://cloud.google.com/sdk/docs/downloads-interactive)
- OpenAI API 密钥
- ChatGPT Plus、Teams 或 Enterprise 订阅


## 架构

下图展示了该解决方案的架构，我们将逐步进行讲解：

![bigquery-rag-architecture.png](../../../../images/bigquery_rag_architecture.png)

## 目录

1. **[设置环境](#设置环境)** 通过安装和导入所需的库并配置 GCP 设置来准备环境。包括：
    - [安装并导入所需的库](#安装并导入所需的库)
    - [配置 GCP 项目](#配置-gcp-项目)
    - [配置 OpenAI 设置](#配置-openai-设置)


2. **[准备数据](#准备数据)** 通过对文档进行嵌入并收集额外的元数据，为上传准备数据。我们将使用 OpenAI 文档的一个子集作为示例数据。

3. **[创建支持向量搜索的 BigQuery 表](#创建支持向量搜索的-bigquery-表)**  
创建一个 BigQuery 表并上传我们准备好的数据。包括：

    - [创建数据集](#创建-bigquery-数据集)：在 BigQuery 中创建数据集的步骤。
    - [创建表并上传数据](#创建表并上传数据)：在 BigQuery 中创建表的说明。

4. **[创建 GCP 函数](#创建-gcp-函数)**，使用 gcloud CLI 和之前计算出的环境变量

5. **[在 ChatGPT 的自定义 GPT 中接入](#在-chatgpt-的自定义-gpt-中接入)** 对 BigQuery 中已嵌入的数据执行搜索：

    - [向量搜索](#测试搜索)：执行基于向量的搜索查询的步骤。
    - [元数据过滤搜索](#使用元数据过滤执行搜索)：执行元数据过滤的说明。

# 设置环境

## 安装并导入所需的库
下面的库可以分为 Python 标准库、第三方库以及 GCP 相关库。

In [ ]:
! pip install -q google-auth
! pip install -q openai
! pip install -q pandas
! pip install -q google-cloud-functions
! pip install -q python-dotenv
! pip install -q pyperclip
! pip install -q PyPDF2
! pip install -q tiktoken
! pip install -q google-cloud-bigquery
! pip install -q pyyaml

In [ ]:
# Standard Libraries
import json  
import os
import csv
import shutil
from itertools import islice
import concurrent.futures
import yaml

# Third-Party Libraries
import pandas as pd
import numpy as np
from PyPDF2 import PdfReader
import tiktoken
from dotenv import load_dotenv
import pyperclip

# OpenAI Libraries
from openai import OpenAI

# Google Cloud Identity and Credentials
from google.auth import default
from google.cloud import bigquery
from google.cloud import functions_v1


## 配置 GCP 项目

如果尚未设置，我们将安装 GCP CLI、完成 GCP 身份验证并设置默认项目。

In [ ]:
# Add gcloud to PATH
os.environ['PATH'] += os.pathsep + os.path.expanduser('~/google-cloud-sdk/bin')

# Verify gcloud is in PATH
! gcloud --version

In [ ]:
! gcloud auth application-default login

In [ ]:
project_id = "<insert_project_id>"  # Replace with your actual project ID
! gcloud config set project {project_id}

In [ ]:
! gcloud services enable cloudfunctions.googleapis.com
! gcloud services enable cloudbuild.googleapis.com
! gcloud services enable bigquery.googleapis.com

## 配置 OpenAI 设置

本节将指导你设置 OpenAI 的身份验证。开始本节之前，请确保你已准备好 OpenAI API 密钥。

In [ ]:
openai_api_key = os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as an env var>") # Saving this as a variable to reference in function app in later step
openai_client = OpenAI(api_key=openai_api_key)
embeddings_model = "text-embedding-3-small" # We'll use this by default, but you can change to your text-embedding-3-large if desired

## 配置具备向量搜索能力的 GCP BigQuery

本节说明如何在 BigQuery 中创建数据集，并存储用于嵌入和向量搜索的浮点向量。

In [ ]:
from google.auth import default

# Use default credentials
credentials, project_id = default()
region = "us-central1" # e.g: "us-central1"
print("Default Project ID:", project_id)

# 准备数据
我们将对 oai_docs 文件夹中的若干 OpenAI 文档页面进行嵌入和存储。我们会先对每个页面进行嵌入并添加到 CSV 中，然后使用该 CSV 上传到索引。

我们将使用[这篇 cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/Embedding_long_inputs.ipynb) 中介绍的一些技巧。这是一种快速嵌入文本的方法，没有考虑诸如章节划分、使用视觉模型描述图像/图表/示意图、为较长文档在分块之间保留重叠文本等因素。

为了处理超出 8191 个 token 上下文长度的较长文本文件，我们可以分别使用各个分块的嵌入，或者以某种方式将它们组合起来，例如求平均值（按每个分块的大小加权）。

我们将借用 Python 官方 cookbook 中的一个函数，用于将序列切分为多个分块。

In [ ]:
def batched(iterable, n):
    """Batch data into tuples of length n. The last batch may be shorter."""
    # batched('ABCDEFG', 3) --> ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while (batch := tuple(islice(it, n))):
        yield batch


现在我们定义一个函数，将字符串编码为 token，再将其切分为分块。我们将使用 tiktoken，这是 OpenAI 推出的一款快速开源分词器。

要进一步了解如何使用 Tiktoken 计算 token，请参阅[这篇 cookbook](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken)。



In [ ]:
def chunked_tokens(text, chunk_length, encoding_name='cl100k_base'):
    # Get the encoding object for the specified encoding name. OpenAI's tiktoken library, which is used in this notebook, currently supports two encodings: 'bpe' and 'cl100k_base'. The 'bpe' encoding is used for GPT-3 and earlier models, while 'cl100k_base' is used for newer models like GPT-4.
    encoding = tiktoken.get_encoding(encoding_name)
    # Encode the input text into tokens
    tokens = encoding.encode(text)
    # Create an iterator that yields chunks of tokens of the specified length
    chunks_iterator = batched(tokens, chunk_length)
    # Yield each chunk from the iterator
    yield from chunks_iterator

最后，我们可以编写一个函数，通过将输入 token 分块并分别对每个分块进行嵌入，即使输入文本超过最大上下文长度，也能安全地处理嵌入请求。将 average 标志设为 True 会返回各分块嵌入的加权平均值，设为 False 则直接返回未经修改的分块嵌入列表。

> 注意：这里还可以采用其他技巧，包括：
> - 使用 GPT-4o 获取图像/图表的描述用于嵌入
> - 基于段落或章节进行分块
> - 为每篇文章添加更具描述性的元数据。

In [ ]:
EMBEDDING_CTX_LENGTH = 8191
EMBEDDING_ENCODING='cl100k_base'

In [ ]:
def generate_embeddings(text, model):
    # Generate embeddings for the provided text using the specified model
    embeddings_response = openai_client.embeddings.create(model=model, input=text)
    # Extract the embedding data from the response
    embedding = embeddings_response.data[0].embedding
    return embedding

def len_safe_get_embedding(text, model=embeddings_model, max_tokens=EMBEDDING_CTX_LENGTH, encoding_name=EMBEDDING_ENCODING):
    # Initialize lists to store embeddings and corresponding text chunks
    chunk_embeddings = []
    chunk_texts = []
    # Iterate over chunks of tokens from the input text
    for chunk in chunked_tokens(text, chunk_length=max_tokens, encoding_name=encoding_name):
        # Generate embeddings for each chunk and append to the list
        chunk_embeddings.append(generate_embeddings(chunk, model=model))
        # Decode the chunk back to text and append to the list
        chunk_texts.append(tiktoken.get_encoding(encoding_name).decode(chunk))
    # Return the list of chunk embeddings and the corresponding text chunks
    return chunk_embeddings, chunk_texts

接下来，我们可以定义一个辅助函数，用于收集有关文档的额外元数据。在本示例中，我将从一个类别列表中进行选择，供后续在元数据过滤中使用

In [ ]:
categories = ['authentication','models','techniques','tools','setup','billing_limits','other']

def categorize_text(text, categories):

    # Create a prompt for categorization
    messages = [
        {"role": "system", "content": f"""You are an expert in LLMs, and you will be given text that corresponds to an article in OpenAI's documentation.
         Categorize the document into one of these categories: {', '.join(categories)}. Only respond with the category name and nothing else."""},
        {"role": "user", "content": text}
    ]
    try:
        # Call the OpenAI API to categorize the text
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )

        # Extract the category from the response
        category = response.choices[0].message.content
        return category
    except Exception as e:
        print(f"Error categorizing text: {str(e)}")
        return None

# Example usage

现在，我们可以定义一些辅助函数来处理 oai_docs 文件夹中的 .txt 文件。你也可以将其用于自己的数据，它同时支持 .txt 和 .pdf 文件。

In [ ]:
def extract_text_from_pdf(pdf_path):
    # Initialize the PDF reader
    reader = PdfReader(pdf_path)
    text = ""
    # Iterate through each page in the PDF and extract text
    for page in reader.pages:
        text += page.extract_text()
    return text

def process_file(file_path, idx, categories, embeddings_model):
    file_name = os.path.basename(file_path)
    print(f"Processing file {idx + 1}: {file_name}")
    
    # Read text content from .txt files
    if file_name.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
    # Extract text content from .pdf files
    elif file_name.endswith('.pdf'):
        text = extract_text_from_pdf(file_path)
    
    title = file_name
    # Generate embeddings for the title
    title_vectors, title_text = len_safe_get_embedding(title, embeddings_model)
    print(f"Generated title embeddings for {file_name}")
    
    # Generate embeddings for the content
    content_vectors, content_text = len_safe_get_embedding(text, embeddings_model)
    print(f"Generated content embeddings for {file_name}")
    
    category = categorize_text(' '.join(content_text), categories)
    print(f"Categorized {file_name} as {category}")
    
    # Prepare the data to be appended
    data = []
    for i, content_vector in enumerate(content_vectors):
        data.append({
            "id": f"{idx}_{i}",
            "vector_id": f"{idx}_{i}",
            "title": title_text[0],
            "text": content_text[i],
            "title_vector": json.dumps(title_vectors[0]),  # Assuming title is short and has only one chunk
            "content_vector": json.dumps(content_vector),
            "category": category
        })
        print(f"Appended data for chunk {i + 1}/{len(content_vectors)} of {file_name}")
    
    return data



现在我们将使用这个辅助函数来处理 OpenAI 文档。你可以修改下面 process_files 中的文件夹，改用你自己的数据。

请注意，这会并发处理所选文件夹中的文档，因此如果使用 txt 文件，耗时应少于 30 秒；如果使用 PDF，则会稍长一些。

In [ ]:
## Customize the location below if you are using different data besides the OpenAI documentation. Note that if you are using a different dataset, you will need to update the categories list as well.
folder_name = "../../../data/oai_docs"

files = [os.path.join(folder_name, f) for f in os.listdir(folder_name) if f.endswith('.txt') or f.endswith('.pdf')]
data = []

# Process each file concurrently
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(process_file, file_path, idx, categories, embeddings_model): idx for idx, file_path in enumerate(files)}
    for future in concurrent.futures.as_completed(futures):
        try:
            result = future.result()
            data.extend(result)
        except Exception as e:
            print(f"Error processing file: {str(e)}")

# Write the data to a CSV file
csv_file = os.path.join("..", "embedded_data.csv")
with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ["id", "vector_id", "title", "text", "title_vector", "content_vector","category"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in data:
        writer.writerow(row)
        print(f"Wrote row with id {row['id']} to CSV")

# Convert the CSV file to a Dataframe
article_df = pd.read_csv("../embedded_data.csv")
# Read vectors from strings back into a list using json.loads
article_df["title_vector"] = article_df.title_vector.apply(json.loads)
article_df["content_vector"] = article_df.content_vector.apply(json.loads)
article_df["vector_id"] = article_df["vector_id"].apply(str)
article_df["category"] = article_df["category"].apply(str)
article_df.head()

现在我们得到了一个包含六列的 `embedded_data.csv` 文件，可以将其上传到向量数据库了！

# 创建支持向量搜索的 BigQuery 表

## 创建 BigQuery 数据集

我们将借助 Google SDK 创建一个名为 "oai_docs" 的数据集，表名为 "embedded_data"，你也可以随意修改这些变量（还可以更改区域）。

*附注：我们不会创建 BigQuery 索引。虽然索引可以提升向量搜索的性能，但它要求数据集中有超过 1k 行，而我们的示例中没有这么多数据；你可以在自己的用例中加以利用。*

In [ ]:
# Create bigquery table

from google.cloud import bigquery
from google.api_core.exceptions import Conflict

# Define the dataset ID (project_id.dataset_id)
raw_dataset_id = 'oai_docs'
dataset_id = project_id + '.' + raw_dataset_id

client = bigquery.Client(credentials=credentials, project=project_id)

# Construct a full Dataset object to send to the API
dataset = bigquery.Dataset(dataset_id)

# Specify the geographic location where the dataset should reside
dataset.location = "US"

# Send the dataset to the API for creation
try:
    dataset = client.create_dataset(dataset, timeout=30)
    print(f"Created dataset {client.project}.{dataset.dataset_id}")
except Conflict:
    print(f"dataset {dataset.dataset_id } already exists")
    

In [ ]:
# Read the CSV file, properly handling multiline fields
csv_file_path = "../embedded_data.csv"
df = pd.read_csv(csv_file_path, engine='python', quotechar='"', quoting=1)

# Display the first few rows of the dataframe
df.head()


## 创建表并上传数据

我们将使用属性名称和类型创建表。注意 'content_vector' 属性，它允许为单行存储一个浮点向量，我们将用它进行向量搜索。

然后，这段代码会遍历之前创建的 CSV，将各行插入 BigQuery。
如果多次运行这段代码，会插入多条完全相同的行，导致搜索结果不够准确（你可以为 ID 设置唯一性约束，或每次清空数据库）。

In [ ]:
# Read the CSV file, properly handling multiline fields
dataset_id = project_id + '.' + raw_dataset_id
client = bigquery.Client(credentials=credentials, project=project_id)
csv_file_path = "../embedded_data.csv"
df = pd.read_csv(csv_file_path, engine='python', quotechar='"', quoting=1)

# Preprocess the data to ensure content_vector is correctly formatted
# removing last and first character which are brackets [], comma splitting and converting to float
def preprocess_content_vector(row):
    row['content_vector'] = [float(x) for x in row['content_vector'][1:-1].split(',')]
    return row

# Apply preprocessing to the dataframe
df = df.apply(preprocess_content_vector, axis=1)

# Define the schema of the final table
final_schema = [
    bigquery.SchemaField("id", "STRING"),
    bigquery.SchemaField("vector_id", "STRING"),
    bigquery.SchemaField("title", "STRING"),
    bigquery.SchemaField("text", "STRING"),
    bigquery.SchemaField("title_vector", "STRING"),
    bigquery.SchemaField("content_vector", "FLOAT64", mode="REPEATED"),
    bigquery.SchemaField("category", "STRING"),
]

# Define the final table ID
raw_table_id = 'embedded_data'
final_table_id = f'{dataset_id}.' + raw_table_id

# Create the final table object
final_table = bigquery.Table(final_table_id, schema=final_schema)

# Send the table to the API for creation
final_table = client.create_table(final_table, exists_ok=True)  # API request
print(f"Created final table {project_id}.{final_table.dataset_id}.{final_table.table_id}")

# Convert DataFrame to list of dictionaries for BigQuery insertion
rows_to_insert = df.to_dict(orient='records')

# Upload data to the final table
errors = client.insert_rows_json(f"{final_table.dataset_id}.{final_table.table_id}", rows_to_insert)  # API request

if errors:
    print(f"Encountered errors while inserting rows: {errors}")
else:
    print(f"Successfully loaded data into {dataset_id}:{final_table_id}")

# 测试搜索
数据上传完成后，我们将在下方本地测试纯向量相似度搜索以及带元数据过滤的搜索，确保其按预期工作。

你可以同时测试纯向量搜索和元数据过滤。

下面的查询是纯向量搜索，我们不按类别进行过滤。

In [ ]:
query = "What model should I use to embed?"
category = "models"

embedding_query = generate_embeddings(query, embeddings_model)
embedding_query_list = ', '.join(map(str, embedding_query))

query = f"""
WITH search_results AS (
  SELECT query.id AS query_id, base.id AS base_id, distance
  FROM VECTOR_SEARCH(
    TABLE oai_docs.embedded_data, 'content_vector',
    (SELECT ARRAY[{embedding_query_list}] AS content_vector, 'query_vector' AS id),
    top_k => 2, distance_type => 'COSINE', options => '{{"use_brute_force": true}}')
)
SELECT sr.query_id, sr.base_id, sr.distance, ed.text, ed.title
FROM search_results sr
JOIN oai_docs.embedded_data ed ON sr.base_id = ed.id
ORDER BY sr.distance ASC
"""

query_job = client.query(query)
results = query_job.result()  # Wait for the job to complete

for row in results:
    print(f"query_id: {row['query_id']}, base_id: {row['base_id']}, distance: {row['distance']}, text_truncated: {row['text'][0:100]}")


## 使用元数据过滤执行搜索
元数据过滤可以在获取向量搜索中语义最接近的结果的基础上，进一步将结果限制为具有特定属性的条目。

下面的代码演示了如何执行带元数据过滤的查询：

In [ ]:

query = "What model should I use to embed?"
category = "models"

embedding_query = generate_embeddings(query, embeddings_model)
embedding_query_list = ', '.join(map(str, embedding_query))


query = f"""
WITH search_results AS (
  SELECT query.id AS query_id, base.id AS base_id, distance
  FROM VECTOR_SEARCH(
    (SELECT * FROM oai_docs.embedded_data WHERE category = '{category}'), 
    'content_vector',
    (SELECT ARRAY[{embedding_query_list}] AS content_vector, 'query_vector' AS id),
    top_k => 4, distance_type => 'COSINE', options => '{{"use_brute_force": true}}')
)
SELECT sr.query_id, sr.base_id, sr.distance, ed.text, ed.title, ed.category
FROM search_results sr
JOIN oai_docs.embedded_data ed ON sr.base_id = ed.id
ORDER BY sr.distance ASC
"""


query_job = client.query(query)
results = query_job.result()  # Wait for the job to complete

for row in results:
    print(f"category: {row['category']}, title: {row['title']}, base_id: {row['base_id']}, distance: {row['distance']}, text_truncated: {row['text'][0:100]}")


# 创建 GCP 函数

## 导出变量

我们将部署此文件夹中 `main.py` 里的函数（也可以在[这里](https://github.com/openai/openai-cookbook/blob/main/examples/chatgpt/rag-quickstart/gcp/main.py)获取）。

第一步，我们会导出变量，用于指定目标表/数据集，以及使用 OpenAI 的 API 生成嵌入。

In [ ]:
# Create a dictionary to store the environment variables (they were used previously and are just retrieved)
env_variables = {
    'OPENAI_API_KEY': openai_api_key,
    'EMBEDDINGS_MODEL': embeddings_model,
    'PROJECT_ID': project_id,
    'DATASET_ID': raw_dataset_id,
    'TABLE_ID': raw_table_id
}

# Write the environment variables to a YAML file
with open('env.yml', 'w') as yaml_file:
    yaml.dump(env_variables, yaml_file, default_flow_style=False)

print("env.yml file created successfully.")

## 部署函数

现在我们将为当前项目创建一个名为 "openai_docs_search" 的 Google 函数，为此我们会利用之前创建的环境变量，运行下面的 CLI 命令。请注意，该函数可以在任何地方无需身份验证即可调用，请不要将其用于生产环境，或者添加额外的身份验证机制。

In [ ]:
! gcloud functions deploy openai_docs_search \
  --runtime python39 \
  --trigger-http \
  --allow-unauthenticated \
  --env-vars-file env.yml

# 在 ChatGPT 的自定义 GPT 中接入

现在我们已经有了一个查询该向量搜索索引的 GCP Function，接下来把它设置为 GPT Action！

关于 GPT，请参阅[这里](https://openai.com/index/introducing-gpts/)的文档；关于 GPT Actions，请参阅[这里](https://platform.openai.com/docs/actions)。将下面的内容分别用作 GPT 的指令和 GPT Action 的 OpenAPI 规范。

## 创建 OpenAPI 规范

下面是一个 OpenAPI 规范示例。运行下面的代码块后，一份可用的规范会被复制到剪贴板，供你粘贴到 GPT Action 中。

请注意，默认情况下它没有任何身份验证，但你可以按照[这里](https://cloud.google.com/functions/docs/securing/authenticating)的 GCP 文档为 GCP Functions 设置身份验证。

In [ ]:
spec = f"""
openapi: 3.1.0
info:
  title: OpenAI API documentation search
  description: API to perform a semantic search over OpenAI APIs
  version: 1.0.0
servers:
  - url: https://{region}-{project_id}.cloudfunctions.net
    description: Main (production) server
paths:
  /openai_docs_search:
    post:
      operationId: openai_docs_search
      summary: Perform a search
      description: Returns search results for the given query parameters.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                query:
                  type: string
                  description: The search query string
                top_k:
                  type: integer
                  description: Number of top results to return. Maximum is 3.
                category:
                  type: string
                  description: The category to filter on, on top of similarity search (used for metadata filtering). Possible values are {categories}.
      responses:
        '200':
          description: A JSON response with the search results
          content:
            application/json:
              schema:
                type: object
                properties:
                  items:
                    type: array
                    items:
                      type: object
                      properties:
                        text:
                          type: string
                          example: "Learn how to turn text into numbers, unlocking use cases like search..."
                        title:
                          type: string
                          example: "embeddings.txt"
                        distance:
                          type: number
                          format: float
                          example: 0.484939891778730
                        category:
                          type: string
                          example: "models"
"""
print(spec)
pyperclip.copy(spec)
print("OpenAPI spec copied to clipboard")


## 创建 GPT 指令

你可以根据需要随意修改指令。关于提示词工程的一些技巧，请查看我们[这里](https://platform.openai.com/docs/guides/prompt-engineering)的文档。

In [ ]:
instructions = f'''
You are an OpenAI docs assistant. You have an action in your knowledge base where you can make a POST request to search for information. The POST request should always include: {{
    "query": "<user_query>",
    "k_": <integer>,
    "category": <string, but optional>
}}. Your goal is to assist users by performing searches using this POST request and providing them with relevant information based on the query.

You must only include knowledge you get from your action in your response.
The category must be from the following list: {categories}, which you should determine based on the user's query. If you cannot determine, then do not include the category in the POST request.
'''
pyperclip.copy(instructions)
print("GPT Instructions copied to clipboard")
print(instructions)

# 回顾

通过以下步骤，我们已经成功地将 GCP BigQuery 向量搜索与 ChatGPT 中的 GPT Actions 集成：
1. 使用 OpenAI 的嵌入对文档进行嵌入，同时使用 gpt-4o 添加一些额外的元数据。
2. 将这些数据上传到 GCP BigQuery（原始数据和嵌入向量）
3. 在 GCP Functions 上创建一个用于检索这些数据的端点
4. 将其整合到自定义 GPT 中。

现在，我们的 GPT 可以检索信息来帮助回答用户的问题，使回答更加准确，并且更贴合我们的数据。下面是 GPT 的实际运行效果：

![gcp-rag-quickstart-gpt.png](../../../../images/gcp_rag_quickstart_gpt.png)